In [1]:
from pathlib import Path

from maite_datasets.object_detection import SkySeaLand

data_root = Path("./data")

# One ~262 MB download into ./data/skysealand, shared by all three exports below.
# A re-run reads what is already on disk instead of downloading again.
SkySeaLand(root=data_root, image_set="base", download=True)

split_paths = {name: data_root / f"skysealand_datamaite_{name}" for name in ("train", "val", "test")}
for image_set in split_paths:
    SkySeaLand(root=data_root, image_set=image_set, as_datamaite=True)

print("\n".join(f"{name}: {path}" for name, path in split_paths.items()))

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train: data/skysealand_datamaite_train
val: data/skysealand_datamaite_val
test: data/skysealand_datamaite_test


In [2]:
from dataeval.config import set_max_processes

from dataeval_flow import PipelineConfig, run_task
from dataeval_flow.config import (
    CocoDatasetConfig,
    MetadataPolicyConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.workflows.data_analysis import DataAnalysisConfig, DataAnalysisHealthThresholds

# Limit concurrency to 4 processes for memory management during image decoding.
set_max_processes(4)

analysis_workflow = DataAnalysisConfig(
    name="skysealand_analysis",
    outlier_method="adaptive",
    outlier_flags=["dimension", "pixel", "visual"],
    outlier_threshold=4.0,
    balance=True,
    diversity_method="simpson",
    metadata="skysealand_factors",
    health_thresholds=DataAnalysisHealthThresholds(
        image_outliers=5.0,  # Relaxed from 3% for diverse overhead imagery
        exact_duplicates=0.0,  # No exact duplicates allowed (default)
        near_duplicates=5.0,  # Up to 5% near duplicates before warning (default)
        class_label_imbalance=5.0,  # Default; SkySeaLand sits near 2.5:1 in every split
        distribution_shift=0.5,  # Default
    ),
)

task = TaskConfig(
    name="skysealand-quality-check",
    workflow="skysealand_analysis",
    sources=["train", "val", "test"],
)

config = PipelineConfig(
    metadata=[
        MetadataPolicyConfig(
            name="skysealand_factors",
            # Image statistics evaluated as factors for bias analysis
            intrinsic_factors=["visual", "pixel"],
            # Exclude constant metadata fields
            exclude=["label_file_exists"],
            # Shared encoding reference for cross-split factor comparability
            reference_split="train",
        )
    ],
    datasets=[CocoDatasetConfig(name=f"skysealand_{name}", path=str(path)) for name, path in split_paths.items()],
    views=[
        ViewConfig(
            name="sample300",
            operations=[
                ViewOperation(type="Shuffle", params={"seed": 0}),
                ViewOperation(type="Limit", params={"size": 300}),
            ],
        ),
    ],
    sources=[
        SourceConfig(name="train", dataset="skysealand_train", view="sample300"),
        SourceConfig(name="val", dataset="skysealand_val"),
        SourceConfig(name="test", dataset="skysealand_test"),
    ],
    workflows=[analysis_workflow],
    tasks=[task],
)

print("Configuration ready:")
print(f"  Workflow:   {analysis_workflow.name} (type={analysis_workflow.type})")
print(f"  Task:       {task.name} -> {task.workflow}")
print(f"  Sources:    {task.sources}")

Configuration ready:
  Workflow:   skysealand_analysis (type=data-analysis)
  Task:       skysealand-quality-check -> skysealand_analysis
  Sources:    ['train', 'val', 'test']


In [3]:
result = run_task(task, config, cache_dir=Path("./cache"))

/builds/jatic/aria/dataeval-flow/src/dataeval_flow/_binning.py:152: UserWarning: `height`, `instance_brightness`, `instance_contrast`, `instance_darkness`, `instance_entropy`, `instance_kurtosis`, `instance_mean`, `instance_sharpness`, `instance_skew`, `instance_std`, `instance_var`, `instance_zeros`, `num_annotations`, `unit_brightness`, `unit_contrast`, `unit_darkness`, `unit_entropy`, `unit_kurtosis`, `unit_mean`, `unit_sharpness`, `unit_skew`, `unit_std`, `unit_var`, `unit_zeros` and `width` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"height": [...]} to control this.
  export(path)


/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.11/site-packages/dataeval/core/_mutual_info.py:91: RuntimeWarning: divide by zero encountered in log
  return float(expected_mutual_information(rescaled(contingency, n_effective), n_effective))


In [4]:
if not result.success:
    print(f"Workflow failed: {result.errors}")
assert result.success

In [5]:
print(result.report())


  DATASET ANALYSIS COMPLETE. 3 SPLIT(S), 559 TOTAL ITEMS.
  Timestamp:    2026-09-25T21:13:03.476646+00:00
  Duration:     16.15s
  Source:       train (skysealand_train[sample300])
                val (skysealand_val)
                test (skysealand_test)
------------------------------------------------------------------------------------------

  SUMMARY
  -------
  Image Quality ............................. 37 outliers (17/300, 10/132, 10/127)  [!!]
  Redundancy .......................................... No duplicates in any split  [ok]
  Label Balance ............................... 4 classes, imbalance 2.6/2.3/2.4:1  [..]
  Bias .......................................... 27 factors checked, issues found  [!!]
  Label Overlap ............................... All 4 classes shared across splits  [ok]
  Label Parity ............................... 3/3 pair(s) significantly different  [!!]
  Leakage .............................................. No cross-split duplicates  [ok]

  Hea

In [6]:
import polars as pl

raw = result.output.raw

for pair_name, comparison in raw.cross_split.items():
    overlap = comparison.label_health.label_overlap

    # Check for split-exclusive classes
    split_only = {k: v for k, v in overlap.items() if k.endswith("_only") and v}
    if split_only:
        print(f"--- {pair_name}: MISSING CLASSES ---")
        for key, val in split_only.items():
            print(f"  {key}: {val}")
    else:
        shared = overlap.get("shared_classes", [])
        print(f"--- {pair_name}: all {len(shared)} classes present in both splits ---")

    # Proportion comparison table
    prop = overlap.get("proportion_comparison", {})
    if prop:
        prop_rows = []
        first = next(iter(prop.values()))
        pair_splits = [k for k in first if k != "difference"]
        for cls_name, vals in prop.items():
            row = {"Class": cls_name}
            for s in pair_splits:
                row[f"{s} (%)"] = round(vals[s] * 100, 1)
            row["Diff (pp)"] = round(vals["difference"] * 100, 1)
            prop_rows.append(row)
        df = pl.DataFrame(prop_rows).sort("Diff (pp)", descending=True)
        large_diffs = [c for c, v in prop.items() if abs(v["difference"]) > 0.05]
        if large_diffs:
            print(f"  {len(large_diffs)} class(es) differ by >5 percentage points between splits")
        print(df)
    print()

--- train_vs_val: all 4 classes present in both splits ---
  3 class(es) differ by >5 percentage points between splits
shape: (4, 4)
┌──────────┬───────────┬─────────┬───────────┐
│ Class    ┆ train (%) ┆ val (%) ┆ Diff (pp) │
│ ---      ┆ ---       ┆ ---     ┆ ---       │
│ str      ┆ f64       ┆ f64     ┆ f64       │
╞══════════╪═══════════╪═════════╪═══════════╡
│ boat     ┆ 15.8      ┆ 33.0    ┆ 17.2      │
│ car      ┆ 41.7      ┆ 34.1    ┆ 7.6       │
│ airplane ┆ 25.8      ┆ 18.4    ┆ 7.4       │
│ ship     ┆ 16.7      ┆ 14.5    ┆ 2.1       │
└──────────┴───────────┴─────────┴───────────┘

--- train_vs_test: all 4 classes present in both splits ---
shape: (4, 4)
┌──────────┬───────────┬──────────┬───────────┐
│ Class    ┆ train (%) ┆ test (%) ┆ Diff (pp) │
│ ---      ┆ ---       ┆ ---      ┆ ---       │
│ str      ┆ f64       ┆ f64      ┆ f64       │
╞══════════╪═══════════╪══════════╪═══════════╡
│ car      ┆ 41.7      ┆ 38.5     ┆ 3.3       │
│ ship     ┆ 16.7      ┆ 18.8     

In [7]:
for pair_name, comparison in raw.cross_split.items():
    lp = comparison.label_health.label_parity
    if lp:
        if lp["significant"]:
            print(
                f"{pair_name}: SIGNIFICANT difference (chi2={lp['chi_squared']:.2f}, "
                f"p={lp['p_value']:.4g}) -- splits may not share the same label distribution"
            )
        else:
            print(f"{pair_name}: no significant difference (chi2={lp['chi_squared']:.2f}, p={lp['p_value']:.4g})")
    else:
        print(f"{pair_name}: label parity not computed")

train_vs_val: SIGNIFICANT difference (chi2=448.88, p=5.709e-97) -- splits may not share the same label distribution
train_vs_test: SIGNIFICANT difference (chi2=11.76, p=0.008237) -- splits may not share the same label distribution
val_vs_test: SIGNIFICANT difference (chi2=293.53, p=2.503e-63) -- splits may not share the same label distribution


In [8]:
for split_name, split_data in raw.splits.items():
    rd = split_data.redundancy
    print(f"{split_name}: {len(rd.exact_groups)} exact, {len(rd.near_groups)} near duplicate group(s)")
    for i, group in enumerate(rd.exact_groups):
        print(f"  exact group {i + 1}: {[f'{split_name}[{idx}]' for idx in group]}")
    for i, group in enumerate(rd.near_groups):
        print(f"  near group {i + 1}: {[f'{split_name}[{idx}]' for idx in group]}")

train: 0 exact, 0 near duplicate group(s)
val: 0 exact, 0 near duplicate group(s)
test: 0 exact, 0 near duplicate group(s)


In [9]:
import matplotlib.pyplot as plt
import numpy as np

assert result.sources is not None

for pair_name, comparison in raw.cross_split.items():
    leakage = comparison.redundancy.duplicate_leakage
    exact_count = leakage.get("exact_count", 0)
    near_count = leakage.get("near_count", 0)
    if exact_count == 0 and near_count == 0:
        print(f"{pair_name}: No cross-split duplicates -- split integrity preserved")
        continue

    print(f"{pair_name}: DATA LEAKAGE DETECTED -- {exact_count} exact, {near_count} near duplicates")

    # Render exact duplicate groups: images from both splits side by side
    for i, group in enumerate(leakage.get("exact_groups", [])):
        all_images = []
        all_labels = []
        for split_name, indices in group.items():
            for idx in indices:
                img = np.array(result.sources[split_name][idx][0])
                if img.ndim == 3 and img.shape[0] in (1, 3, 4):
                    img = img.transpose(1, 2, 0)
                all_images.append(img)
                all_labels.append(f"{split_name}[{idx}]")
        if all_images:
            n = len(all_images)
            fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
            if n == 1:
                axes = [axes]
            for ax, img, label in zip(axes, all_images, all_labels, strict=True):
                ax.imshow(img)
                ax.set_title(label, fontsize=10)
                ax.axis("off")
            fig.suptitle(f"Cross-split exact duplicate group {i + 1}", fontsize=12, fontweight="bold")
            fig.tight_layout()

    # Print near duplicate leakage groups
    for i, group in enumerate(leakage.get("near_groups", [])):
        labels = []
        for split_name, indices in group.items():
            labels.extend(f"{split_name}[{idx}]" for idx in indices)
        if labels:
            print(f"  Near duplicate group {i + 1}: {labels}")

train_vs_val: No cross-split duplicates -- split integrity preserved
train_vs_test: No cross-split duplicates -- split integrity preserved
val_vs_test: No cross-split duplicates -- split integrity preserved


In [10]:
json_str = result.export(fmt="json")
print(f"JSON output: {len(json_str)} characters")
print(json_str[:500] + "\n...")

JSON output: 1198022 characters
{
  "kind": "workflow",
  "metadata": {
    "version": "1.0",
    "timestamp": "2026-09-25T21:13:03.476646Z",
    "dataset_id": "skysealand_train,skysealand_val,skysealand_test",
    "label_source": "annotations",
    "model_id": null,
    "preprocessor_id": null,
    "selection_id": "sample300",
    "source_descriptions": [
      "train (skysealand_train[sample300])",
      "val (skysealand_val)",
      "test (skysealand_test)"
    ],
    "resolved_config": {
      "sources": [
        {
      
...
